# Qwen3-0.6B full DPO on all control datasets — Colab A100 from VS Code

This notebook trains on 4,500 examples and validates on 500 held-out examples (5,000 total): the original all-caps, no-commas, and disclaimer controls plus seven multilingual controls. It uses a 4,096-token cutoff for one epoch, evaluates before and after training, saves the epoch checkpoint to Google Drive, and produces a training-versus-validation preference-accuracy report.

It is designed for VS Code connected to a Colab runtime. Before running, select an A100 GPU in Colab and copy `colab_qwen_dpo_bundle.zip` into the root of Google Drive (`MyDrive`).

In [4]:
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "Enable a GPU runtime first."
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"({props.total_memory / 2**30:.1f} GiB)")
assert "A100" in props.name, "Select an A100 runtime for this notebook."
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)

GPU: NVIDIA A100-SXM4-40GB (39.5 GiB)
PyTorch: 2.11.0+cu128 CUDA: 12.8


## 1. Mount Drive and locate the prepared bundle

Because the kernel is controlled from VS Code, this notebook does not use Colab's browser upload widget. Copy `colab_qwen_dpo_bundle.zip` to Google Drive first.

In [3]:
from google.colab import drive
from pathlib import Path
import shutil, zipfile

drive.mount('/content/drive')
BUNDLE_IN_DRIVE = Path('/content/drive/MyDrive/colab_qwen_dpo_bundle.zip')
assert BUNDLE_IN_DRIVE.is_file(), f"Copy the bundle to Google Drive first: {BUNDLE_IN_DRIVE}"
bundle = BUNDLE_IN_DRIVE

PROJECT = Path('/content/CoT_Controllability')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(PROJECT)
(PROJECT / 'data').mkdir(exist_ok=True)
(PROJECT / 'scripts').mkdir(exist_ok=True)
for json_file in PROJECT.glob('*.json'):
    shutil.move(json_file, PROJECT / 'data' / json_file.name)
for script_file in PROJECT.glob('*.py'):
    shutil.move(script_file, PROJECT / 'scripts' / script_file.name)
print("Extracted:", PROJECT)
print("Source bytes:", (PROJECT / 'data/multilingual_thinking_qwen3_4b_cots.json').stat().st_size)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracted: /content/CoT_Controllability
Source bytes: 13252140


## 2. Install the training stack

This keeps Colab's CUDA-enabled PyTorch and installs LLaMA-Factory plus TensorBoard.

In [5]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'llamafactory[torch,metrics]==0.9.5',
    'transformers>=4.51,<4.57', 'datasets>=3.2', 'accelerate>=1.2',
    'sentencepiece', 'protobuf', 'tensorboard', 'matplotlib', 'pandas'
], check=True)
import llamafactory, transformers
print("LLaMA-Factory:", llamafactory.__version__)
print("Transformers:", transformers.__version__)

LLaMA-Factory: 0.9.5
Transformers: 4.56.2


## 3. Rebuild, validate, and register every preference dataset

In [6]:
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/prepare_all_dpo_data.py'),
    '--input', str(PROJECT / 'data/multilingual_thinking_qwen3_4b_cots.json'),
    '--output-dir', str(PROJECT / 'data'), '--eval-size', '100'
], check=True)

import json, math
registry = json.loads((PROJECT / 'data/dataset_info.json').read_text(encoding='utf-8'))
train_names = sorted(name for name in registry if name.endswith('_train'))
eval_names = sorted(name for name in registry if name.endswith('_eval'))
assert len(train_names) == len(eval_names) == 10
train_count = sum(len(json.loads((PROJECT / 'data' / registry[name]['file_name']).read_text(encoding='utf-8'))) for name in train_names)
eval_count = sum(len(json.loads((PROJECT / 'data' / registry[name]['file_name']).read_text(encoding='utf-8'))) for name in eval_names)
assert train_count == 4500 and eval_count == 500
steps_per_epoch = math.ceil(train_count / (1 * 4))
assert steps_per_epoch == 1125
print('Training datasets:', train_names)
print('Validation datasets:', eval_names)
print(f'Total: {train_count} train + {eval_count} validation = {train_count + eval_count}')
print(f'Optimizer steps: {steps_per_epoch} total for one epoch')

Training datasets: ['all_caps_dpo_train', 'disclaimer_at_end_dpo_train', 'multilingual_ar_dpo_train', 'multilingual_en_dpo_train', 'multilingual_es_dpo_train', 'multilingual_fr_dpo_train', 'multilingual_hi_dpo_train', 'multilingual_ru_dpo_train', 'multilingual_zh-Hans_dpo_train', 'no_commas_dpo_train']
Validation datasets: ['all_caps_dpo_eval', 'disclaimer_at_end_dpo_eval', 'multilingual_ar_dpo_eval', 'multilingual_en_dpo_eval', 'multilingual_es_dpo_eval', 'multilingual_fr_dpo_eval', 'multilingual_hi_dpo_eval', 'multilingual_ru_dpo_eval', 'multilingual_zh-Hans_dpo_eval', 'no_commas_dpo_eval']
Total: 4500 train + 500 validation = 5000
Optimizer steps: 1125 total for one epoch


## 4. Write the A100 training configuration

The effective batch size remains four: micro-batch 1 × gradient accumulation 4. This reduces A100 memory pressure at the 4,096-token cutoff. Checkpoints and metrics go directly to Drive.

In [7]:
import yaml
DRIVE_OUTPUT = Path('/content/drive/MyDrive/CoT_Controllability/qwen3-0.6b-all-controls-5k-cutoff4096-1epoch')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

config = {
    'stage': 'dpo', 'do_train': True, 'finetuning_type': 'full',
    'pref_beta': 0.1, 'pref_loss': 'sigmoid',
    'model_name_or_path': 'Qwen/Qwen3-0.6B',
    'dataset_dir': str(PROJECT / 'data'),
    'dataset': ','.join(train_names),
    'eval_dataset': ','.join(eval_names),
    'template': 'qwen3', 'cutoff_len': 4096,
    'overwrite_cache': True, 'preprocessing_num_workers': 4,
    'output_dir': str(DRIVE_OUTPUT), 'overwrite_output_dir': False,
    'report_to': 'tensorboard', 'logging_steps': 1, 'disable_tqdm': True,
    'per_device_train_batch_size': 1, 'per_device_eval_batch_size': 1,
    'gradient_accumulation_steps': 4, 'gradient_checkpointing': True,
    'bf16': True, 'tf32': True, 'learning_rate': 1e-6,
    'num_train_epochs': 1.0, 'lr_scheduler_type': 'cosine',
    'warmup_ratio': 0.05, 'max_grad_norm': 1.0,
    'eval_strategy': 'epoch', 'eval_on_start': True,
    'save_strategy': 'epoch', 'save_total_limit': 3,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_rewards/accuracies', 'greater_is_better': True,
    'seed': 42, 'data_seed': 42,
}

checkpoints = sorted(DRIVE_OUTPUT.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]))
if checkpoints:
    config['resume_from_checkpoint'] = str(checkpoints[-1])
    print('Will resume from:', checkpoints[-1])
CONFIG_PATH = PROJECT / 'configs/colab_all_datasets_2epochs.yaml'
CONFIG_PATH.parent.mkdir(exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(CONFIG_PATH.read_text())

stage: dpo
do_train: true
finetuning_type: full
pref_beta: 0.1
pref_loss: sigmoid
model_name_or_path: Qwen/Qwen3-0.6B
dataset_dir: /content/CoT_Controllability/data
dataset: all_caps_dpo_train,disclaimer_at_end_dpo_train,multilingual_ar_dpo_train,multilingual_en_dpo_train,multilingual_es_dpo_train,multilingual_fr_dpo_train,multilingual_hi_dpo_train,multilingual_ru_dpo_train,multilingual_zh-Hans_dpo_train,no_commas_dpo_train
eval_dataset: all_caps_dpo_eval,disclaimer_at_end_dpo_eval,multilingual_ar_dpo_eval,multilingual_en_dpo_eval,multilingual_es_dpo_eval,multilingual_fr_dpo_eval,multilingual_hi_dpo_eval,multilingual_ru_dpo_eval,multilingual_zh-Hans_dpo_eval,no_commas_dpo_eval
template: qwen3
cutoff_len: 4096
overwrite_cache: true
preprocessing_num_workers: 4
output_dir: /content/drive/MyDrive/CoT_Controllability/qwen3-0.6b-all-controls-5k-cutoff4096-1epoch
overwrite_output_dir: false
report_to: tensorboard
logging_steps: 1
disable_tqdm: true
per_device_train_batch_size: 1
per_device_e

## 5. Train

Baseline validation runs first. With effective batch size four, the run covers 1,125 optimizer steps and saves `checkpoint-1125`. If Colab disconnects after that checkpoint, reconnect from VS Code and rerun the notebook; the configuration cell automatically detects the completed checkpoint.

In [11]:
import json, os, subprocess, sys, time

env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
env['TOKENIZERS_PARALLELISM'] = 'false'
stdout_path = DRIVE_OUTPUT / 'training_stdout.log'
stderr_path = DRIVE_OUTPUT / 'training_stderr.log'
progress_path = DRIVE_OUTPUT / 'trainer_log.jsonl'

# Re-running this cell reconnects to a process started earlier in this kernel.
if 'training_process' not in globals() or training_process.poll() is not None:
    training_stdout = stdout_path.open('a', encoding='utf-8')
    training_stderr = stderr_path.open('a', encoding='utf-8')
    training_process = subprocess.Popen(
        [sys.executable, '-m', 'llamafactory.cli', 'train', str(CONFIG_PATH)],
        cwd=PROJECT, env=env, stdout=training_stdout, stderr=training_stderr,
    )
    print(f'Started training PID {training_process.pid}', flush=True)
else:
    print(f'Reconnected to training PID {training_process.pid}', flush=True)
print('Detailed logs:', stderr_path, flush=True)

last_signature = None
while training_process.poll() is None:
    if progress_path.exists():
        lines = [line for line in progress_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        if lines:
            status = json.loads(lines[-1])
            signature = (status.get('current_steps'), status.get('eval_loss'), status.get('loss'))
            if signature != last_signature:
                if 'eval_loss' in status:
                    print(
                        f"VALIDATION | step {status.get('current_steps', 0)} | "
                        f"loss {status['eval_loss']:.4f}", flush=True,
                    )
                else:
                    print(
                        f"TRAIN | {status.get('current_steps', 0)}/{status.get('total_steps', 1125)} "
                        f"({status.get('percentage', 0):.2f}%) | "
                        f"loss {status.get('loss', float('nan')):.4f} | "
                        f"accuracy {status.get('accuracy', float('nan')):.1%} | "
                        f"elapsed {status.get('elapsed_time', '?')} | "
                        f"ETA {status.get('remaining_time', '?')}", flush=True,
                    )
                last_signature = signature
    time.sleep(30)

training_stdout.close()
training_stderr.close()
if training_process.returncode != 0:
    tail = stderr_path.read_text(encoding='utf-8', errors='replace').splitlines()[-40:]
    raise RuntimeError('Training failed:\n' + '\n'.join(tail))
print('Training completed successfully.', flush=True)

Started training PID 16874
Detailed logs: /content/drive/MyDrive/CoT_Controllability/qwen3-0.6b-all-controls-5k-cutoff4096-1epoch/training_stderr.log
TRAIN | 1125/1125 (100.00%) | loss nan | accuracy nan% | elapsed 0:47:42 | ETA 0:00:00
Training completed successfully.


## 6. Compare training and validation preference accuracy

In [ ]:
import os
print("Training process running:", process.poll() is None)


NameError: name 'process' is not defined

In [ ]:
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/report_dpo_learning.py'),
    '--output-dir', str(DRIVE_OUTPUT)
], check=True)
from IPython.display import display, Image
display(Image(filename=str(DRIVE_OUTPUT / 'train_vs_validation_accuracy.png')))
print((DRIVE_OUTPUT / 'train_vs_validation_accuracy.csv').read_text())

: 

In [ ]:
from google.colab import runtime
runtime.unassign()

: 

## 7. TensorBoard (optional)

Run the next cell while training or afterward, then open the displayed TensorBoard link.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/CoT_Controllability/qwen3-0.6b-all-controls-5k-cutoff4096-1epoch/runs

: 

: 